| Library           | Single-line use                                                                              |
| ----------------- | -------------------------------------------------------------------------------------------- |
| `datasets`        | Used to load, create, preprocess, and manage datasets for training or fine-tuning models.    |
| `huggingface_hub` | Used to download/upload models, tokenizers, datasets, and checkpoints from Hugging Face Hub. |
| `pillow`          | Used to open, process, resize, and manipulate images in Python.                              |


In [3]:
import unsloth
import os
import torch
from dataclasses import dataclass
from typing import Dict
from datasets import Dataset, Features, Value, Image, load_dataset
from transformers import TextStreamer
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


| Code        | Single-line description                                                                |
| ----------- | -------------------------------------------------------------------------------------- |
| `import os` | Used to work with file paths, folders, and environment variables.                      |
| `Dataset`   | Used to create a Hugging Face dataset from Python data.                                |
| `Features`  | Used to define the schema/structure of dataset columns.                                |
| `Value`     | Used to define text, number, or string column types in the dataset.                    |
| `Image`     | Used to define an image column in the dataset.                                         |
| `login`     | Used to authenticate with Hugging Face Hub for uploading or accessing models/datasets. |


### Images Folder

In [4]:
IMAGE_DIRECTORY = "/content/images"

In [5]:
image_files = sorted([
    os.path.join(IMAGE_DIRECTORY, f)
    for f in os.listdir(IMAGE_DIRECTORY)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
])

In [6]:
image_files

['/content/images/image01.jpg',
 '/content/images/image02.jpg',
 '/content/images/image03.jpg',
 '/content/images/image04.jpg',
 '/content/images/image05.jpg']

### Caption for Images

In [7]:
captions = [
    "An orange iPhone shown from the back a white background.",
    "Three iPhones in white, orange, and dark blue shown together from the back.",
    "An orange iPhone shown from the back and front on a white background.",
    "A dark blue iPhone shown from the back with a side view on a transparent background.",
    "A white iPhone shown from the back and front on a white background.",
]

In [8]:
instruction = "Describe this iPhone product image in one sentence."

In [9]:
rows = []

for img_path, cap in zip(image_files, captions):
    rows.append({
        "image": img_path,
        "text": cap,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": instruction},
                    {"type": "image", "image": img_path},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": cap},
                ],
            },
        ],
    })

In [10]:
rows[0]

{'image': '/content/images/image01.jpg',
 'text': 'An orange iPhone shown from the back a white background.',
 'messages': [{'role': 'user',
   'content': [{'type': 'text',
     'text': 'Describe this iPhone product image in one sentence.'},
    {'type': 'image', 'image': '/content/images/image01.jpg'}]},
  {'role': 'assistant',
   'content': [{'type': 'text',
     'text': 'An orange iPhone shown from the back a white background.'}]}]}

In [11]:
features = Features({
    "image": Image(),
    "text": Value("string"),
    "messages": Value("string"),  # keep as JSON string OR store raw python objects separately
})

| Column                        | Use                                                                             |
| ----------------------------- | ------------------------------------------------------------------------------- |
| `"image": Image()`            | Creates an image column where image files/paths are stored as image data.       |
| `"text": Value("string")`     | Creates a text column for captions, labels, or descriptions.                    |
| `"messages": Value("string")` | Creates a string column to store chat/instruction data |


In [12]:
# If you want to store messages as real objects, easiest is to NOT force Features for messages.
ds = Dataset.from_list(rows)  # messages stored as nested objects

In [13]:
ds_with_features = Dataset.from_list(rows, features=features)

### Train Test Split

In [14]:
splits = ds.train_test_split(test_size=1, seed=3407)

### Push to Hub

In [15]:
from google.colab import userdata

WRITE_TOKEN = userdata.get('HF_TOKEN_WRITE')

In [16]:
REPO_ID = "saadtariq/iphone5_images"

splits["train"].push_to_hub(REPO_ID, split="train", token=WRITE_TOKEN)
splits["test"].push_to_hub(REPO_ID, split="test", token=WRITE_TOKEN)

print("Uploaded:", REPO_ID)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:01<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 4.68kB / 4.68kB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 4.54kB / 4.54kB            

README.md:   0%|          | 0.00/611 [00:00<?, ?B/s]

Uploaded: saadtariq/iphone5_images


In [17]:
REPO_ID = "saadtariq/iphone5_images_with_features"

ds_with_features.push_to_hub(REPO_ID, token=WRITE_TOKEN)

print("Uploaded:", REPO_ID)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 33.4kB / 33.4kB            

README.md:   0%|          | 0.00/335 [00:00<?, ?B/s]

Uploaded: saadtariq/iphone5_images_with_features


In [18]:
 dataset = load_dataset("HuggingFaceM4/ChartQA")

README.md:   0%|          | 0.00/852 [00:00<?, ?B/s]

data/train-00000-of-00003-49492f364babfa(…):   0%|          | 0.00/219M [00:00<?, ?B/s]

data/train-00001-of-00003-7302bae5e425bb(…):   0%|          | 0.00/311M [00:00<?, ?B/s]

data/train-00002-of-00003-194c9400785577(…):   0%|          | 0.00/315M [00:00<?, ?B/s]

data/val-00000-of-00001-0f11003c77497969(…):   0%|          | 0.00/50.2M [00:00<?, ?B/s]

data/test-00000-of-00001-e2cd0b7a0f9eb20(…):   0%|          | 0.00/68.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/28299 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/1920 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [19]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['image', 'query', 'label', 'human_or_machine'],
        num_rows: 28299
    })
    val: Dataset({
        features: ['image', 'query', 'label', 'human_or_machine'],
        num_rows: 1920
    })
    test: Dataset({
        features: ['image', 'query', 'label', 'human_or_machine'],
        num_rows: 2500
    })
})


In [20]:
train_ds = load_dataset("HuggingFaceM4/ChartQA", split="train")

print(train_ds)

Dataset({
    features: ['image', 'query', 'label', 'human_or_machine'],
    num_rows: 28299
})


In [21]:
print(train_ds[0])

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=422x359 at 0x7FB1ABE53CE0>, 'query': 'Is the value of Favorable 38 in 2015?', 'label': ['Yes'], 'human_or_machine': 0}


| Value | Meaning                    |
| ----- | -------------------------- |
| `0`   | Human-generated question   |
| `1`   | Machine-generated question |


## Unsloth Finetuning

In [22]:
# # ================================
# # Qwen 2.5 VL
# # ================================
# model_name = "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit"
# model_name = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit"
# model_name = "unsloth/Qwen2.5-VL-32B-Instruct-bnb-4bit"
# model_name = "unsloth/Qwen2.5-VL-72B-Instruct-bnb-4bit"

# # ================================
# # Qwen 2 VL
# # ================================
# model_name = "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit"
# model_name = "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit"
# model_name = "unsloth/Qwen2-VL-72B-Instruct-bnb-4bit"

# # ================================
# # Qwen 2.5 Omni (Multimodal)
# # ================================
# model_name = "unsloth/Qwen2.5-Omni-3B-Instruct-bnb-4bit"
# model_name = "unsloth/Qwen2.5-Omni-7B-Instruct-bnb-4bit"

# # ================================
# # Llama Vision
# # ================================
# model_name = "unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit"
# model_name = "unsloth/Llama-3.2-90B-Vision-Instruct-bnb-4bit"

# # ================================
# # Gemma Vision
# # ================================
# model_name = "unsloth/MedGemma-4B-Vision-Instruct-bnb-4bit"
# model_name = "unsloth/MedGemma-27B-Vision-Instruct-bnb-4bit"

# # ================================
# # Mistral Vision
# # ================================
# model_name = "unsloth/Pixtral-12B-2409-bnb-4bit"

# # ================================
# # LLaVA
# # ================================
# model_name = "unsloth/llava-1.5-7b-hf-bnb-4bit"
# model_name = "unsloth/llava-v1.6-mistral-7b-hf-bnb-4bit"

### Model Registry

In [23]:
MODEL_REGISTRY = {
    "qwen2_vl_2b": "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
    "qwen25_vl_3b": "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit",
    "llava15_7b": "unsloth/llava-1.5-7b-hf-bnb-4bit",
    "pixtral_12b": "unsloth/Pixtral-12B-2409-bnb-4bit",
    "medgemma_4b": "unsloth/MedGemma-4B-Vision-Instruct-bnb-4bit",
}

### Dataset Registry (Vision-ready datasets)

In [24]:
DATASET_REGISTRY = {
    "latex_ocr": {
        "name": "unsloth/LaTeX_OCR",
        "split": "train",
        "image_key": "image",
        "text_key": "text",
        "instruction": "Write the LaTeX representation for this image."
    },
    "flickr30k": {
        "name": "nlphuji/flickr30k",
        "split": "train",
        "image_key": "image",
        "text_key": "caption",
        "instruction": "Describe the image."
    },

    "iphone_custom": {
    "name": "saadtariq/iphone5_images",  # change to your HF repo
    "split": "train",
    "image_key": "image",
    "text_key": "text",
    "instruction": "Describe this iPhone product image in one sentence."
  },
}

### Config

In [25]:
@dataclass
class VisionFTConfig:
    model_key: str = "qwen2_vl_2b"
    dataset_key: str = "iphone_custom"

    subset_rows: int = 150
    eval_ratio: float = 0.1 #10% data for evaluation
    seed: int = 3407

    # LoRA
    r: int = 16
    lora_alpha: int = 16
    lora_dropout: float = 0.0

    # Training
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    num_train_epochs: int = 2
    learning_rate: float = 2e-4
    logging_steps: int = 10
    weight_decay: float = 0.001
    max_length: int = 2048

    output_dir: str = "outputs"
    save_dir: str = "vlm_lora_output"

| Parameter                              | One-line description                                                    |
| -------------------------------------- | ----------------------------------------------------------------------- |
| `model_key: str = "qwen2_vl_2b"`       | Defines which vision-language model will be used for fine-tuning.       |
| `dataset_key: str = "latex_ocr"`       | Defines which dataset will be used for training.                        |
| `subset_rows: int = 150`               | Uses only 150 rows from the dataset for quick demo/testing.             |
| `eval_ratio: float = 0.1`              | Keeps 10% of the data for evaluation/validation.                        |
| `seed: int = 3407`                     | Fixes randomness so results are reproducible.                           |
| `r: int = 16`                          | LoRA rank; controls how many trainable low-rank parameters are added.   |
| `lora_alpha: int = 16`                 | LoRA scaling factor; controls the strength of LoRA updates.             |
| `lora_dropout: float = 0.0`            | Dropout applied inside LoRA layers to reduce overfitting.               |
| `per_device_train_batch_size: int = 2` | Number of samples processed per GPU/device in one training step.        |
| `gradient_accumulation_steps: int = 4` | Accumulates gradients for 4 steps before updating weights.              |
| `num_train_epochs: int = 2`            | Trains the model for 2 full passes over the training dataset.           |
| `learning_rate: float = 2e-4`          | Controls how fast the model weights are updated during training.        |
| `logging_steps: int = 10`              | Logs training metrics after every 10 steps.                             |
| `weight_decay: float = 0.001`          | Regularization value used to reduce overfitting.                        |
| `max_length: int = 2048`               | Maximum token length allowed for model input/output sequence.           |
| `output_dir: str = "outputs"`          | Folder where training outputs/checkpoints can be saved.                 |
| `save_dir: str = "vlm_lora_output"`    | Final folder where the trained LoRA adapter/model output will be saved. |


### Trainer Class

In [26]:
class VisionFineTuner:
    def __init__(self, cfg: VisionFTConfig):
        self.cfg = cfg

        # Get model and dataset details from registry
        self.model_name = MODEL_REGISTRY[cfg.model_key]
        self.dataset_info = DATASET_REGISTRY[cfg.dataset_key]

        self.model = None
        self.tokenizer = None
        self.train_ds = None
        self.eval_ds = None
        self.trainer = None

    # -----------------------------
    # Load Model + Apply LoRA
    # -----------------------------
    def load_model(self):
        print("Loading model:", self.model_name)

        self.model, self.tokenizer = FastVisionModel.from_pretrained(
            self.model_name,
            load_in_4bit=True,
            use_gradient_checkpointing="unsloth",
        )

        self.model = FastVisionModel.get_peft_model(
            self.model,
            finetune_vision_layers=True,
            finetune_language_layers=True,
            finetune_attention_modules=True,
            finetune_mlp_modules=True,
            r=self.cfg.r,
            lora_alpha=self.cfg.lora_alpha,
            lora_dropout=self.cfg.lora_dropout,
            bias="none",
            random_state=self.cfg.seed,
        )

        print("Model loaded and LoRA applied.")
        return self

    # -----------------------------
    # Prepare Dataset
    # -----------------------------
    def prepare_data(self):
        print("Loading dataset:", self.dataset_info["name"])

        raw = load_dataset(
            self.dataset_info["name"],
            split=self.dataset_info["split"]
        )

        # Use small subset for demo/testing
        raw = raw.select(range(min(self.cfg.subset_rows, len(raw))))

        instruction = self.dataset_info["instruction"]
        image_key = self.dataset_info["image_key"]
        text_key = self.dataset_info["text_key"]

        def format_sample(example):
            return {
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": instruction},
                            {"type": "image", "image": example[image_key]},
                        ],
                    },
                    {
                        "role": "assistant",
                        "content": [
                            {"type": "text", "text": str(example[text_key])}
                        ],
                    },
                ]
            }

        ds = raw.map(
            format_sample,
            remove_columns=raw.column_names
        )

        splits = ds.train_test_split(
            test_size=self.cfg.eval_ratio,
            seed=self.cfg.seed
        )

        self.train_ds = splits["train"]
        self.eval_ds = splits["test"]

        print("Train samples:", len(self.train_ds))
        print("Eval samples:", len(self.eval_ds))

        return self

    # -----------------------------
    # Build Trainer
    # -----------------------------
    def build_trainer(self):
        print("Building trainer...")

        FastVisionModel.for_training(self.model)

        training_args = SFTConfig(
            per_device_train_batch_size=self.cfg.per_device_train_batch_size,
            gradient_accumulation_steps=self.cfg.gradient_accumulation_steps,
            num_train_epochs=self.cfg.num_train_epochs,
            learning_rate=self.cfg.learning_rate,
            logging_steps=self.cfg.logging_steps,
            optim="adamw_8bit",
            weight_decay=self.cfg.weight_decay,
            seed=self.cfg.seed,
            output_dir=self.cfg.output_dir,
            report_to="none",

            # Important for vision-language fine-tuning
            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},

            max_length=self.cfg.max_length,
        )

        self.trainer = SFTTrainer(
            model=self.model,
            tokenizer=self.tokenizer,
            data_collator=UnslothVisionDataCollator(
                self.model,
                self.tokenizer
            ),
            train_dataset=self.train_ds,
            eval_dataset=self.eval_ds,
            args=training_args,
        )

        print("Trainer ready.")
        return self

    # -----------------------------
    # Train Model
    # -----------------------------
    def train(self):
        print("Training started for", self.cfg.num_train_epochs, "epochs")
        self.trainer.train()
        print("Training completed.")
        return self

    # -----------------------------
    # Save Model
    # -----------------------------
    def save(self):
        os.makedirs(self.cfg.save_dir, exist_ok=True)

        self.model.save_pretrained(self.cfg.save_dir)
        self.tokenizer.save_pretrained(self.cfg.save_dir)

        print("Model saved to:", self.cfg.save_dir)
        return self

    # -----------------------------
    # Quick Inference Test
    # -----------------------------
    def quick_infer(self, sample_index=0):
        print("Running quick inference...")

        FastVisionModel.for_inference(self.model)

        raw = load_dataset(
            self.dataset_info["name"],
            split=self.dataset_info["split"]
        )

        image = raw[sample_index][self.dataset_info["image_key"]]

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": self.dataset_info["instruction"]},
                    {"type": "image"},
                ],
            }
        ]

        input_text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = self.tokenizer(
            image,
            input_text,
            add_special_tokens=False,
            return_tensors="pt",
        ).to("cuda")

        streamer = TextStreamer(
            self.tokenizer,
            skip_prompt=True
        )

        self.model.generate(
            **inputs,
            streamer=streamer,
            max_new_tokens=128,
            temperature=1.2,
            do_sample=True,
        )

        return self

    # -----------------------------
    # Full Pipeline Runner
    # -----------------------------
    def run(self):
        self.load_model()
        self.prepare_data()
        self.build_trainer()
        self.train()
        self.save()
        self.quick_infer()

In [28]:
cfg = VisionFTConfig(
    model_key="qwen2_vl_2b",
    dataset_key="iphone_custom",
)

### Create Finetuner Object

In [29]:
trainer = VisionFineTuner(cfg)

#### 1. Load Model and Tokenizer

In [30]:
trainer.load_model()

Loading model: unsloth/Qwen2-VL-2B-Instruct-bnb-4bit
==((====))==  Unsloth 2026.8.8: Fast Qwen2_Vl patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


Model loaded and LoRA applied.


#### 2. Prepare training and evaluation dataset

In [31]:
trainer.prepare_data()

Loading dataset: saadtariq/iphone5_images


README.md:   0%|          | 0.00/611 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/4.68k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/4.54k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Train samples: 3
Eval samples: 1


#### 3. Build SFT Trainer

In [32]:
trainer.build_trainer()

Building trainer...
Unsloth: Model does not have a default image size - using 512
Trainer ready.


#### 4. Train Model

In [33]:
trainer.train()

Training started for 2 epochs


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 2 | Total steps = 2
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 28,950,528 of 2,237,936,128 (1.29% trained)


Step,Training Loss


Training completed.


#### 5. Save Trained LoRA Model and Tokenizer

In [34]:
trainer.save()

Model saved to: vlm_lora_output


#### 6. Test model with one sample image

In [35]:
trainer.quick_infer()

Running quick inference...
The image shows an iPhone product with a white background and a black frame.<|im_end|>
